<a href="https://colab.research.google.com/github/gitmystuff/DTSC3010/blob/main/Week_06/Week06_Distributions_Random_Variables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Probabilities, Random Variables, Sampling, and Empirical Distributions  

Emperical - based on observed data

## Learning objectives
By the end of this notebook, you should be able to:

- Import data from CSV, Excel, and a URL using pandas
- Treat a dataset as a population and draw samples (with and without replacement)
- Build empirical distributions from data and from simulation
- Define a random variable and distinguish discrete vs continuous random variables
- Explain PMF vs PDF vs CDF vs percentiles (PPF)
- Build a sampling distribution by repeated sampling and interpret its center and spread

## In-class norms
- Before running a code cell, read the prompt and make a prediction
- If you get an error, copy the message and identify which line caused it
- Use `Shift+Enter` to run a cell

## Quick warm-up (write your answers here)
- What makes an outcome “random” in a data science context?
- Give one example of a discrete random variable and one example of a continuous random variable.


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

# Reproducibility: one random number generator for the whole notebook
rng = np.random.default_rng(42)

# Display options (optional)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Ready. Libraries imported.")


## 1. Importing data (CSV, Excel, URL)

In many textbooks, datasets appear “magically.” In practice, you usually read data from files or from a web location.

In this section you will:
- Create a small dataset (so this notebook runs anywhere)
- Save it to CSV and Excel
- Read it back from CSV and Excel
- Read an additional dataset from a URL (if internet access is available)

### Your turn (prediction)
Before you run the next cell, answer:
- What do you think `df.shape` returns?
- What do you think `df.describe()` summarizes?


In [ ]:
# Create a small dataset we can use as a "population"
# This avoids file dependency issues while still teaching real import workflows.

n = 500

df_created = pd.DataFrame({
    "total_bill": rng.gamma(shape=2.5, scale=8.0, size=n),      # positive, right-skewed
    "tip": rng.gamma(shape=2.0, scale=1.5, size=n),             # positive, right-skewed
    "party_size": rng.integers(1, 7, size=n),                   # discrete: 1..6
    "day": rng.choice(["Thur", "Fri", "Sat", "Sun"], size=n, p=[0.25, 0.15, 0.40, 0.20]),
})

# Create a derived variable (often done in real analysis)
df_created["tip_pct"] = 100 * df_created["tip"] / df_created["total_bill"]

df_created.head()


### 1A. Save the dataset to CSV and Excel (so we can practice importing)

This simulates a common workflow:
- A dataset exists somewhere (a shared drive, email attachment, exported report)
- You read it into pandas for analysis

We will write two files into your current working directory:
- `example_data.csv`
- `example_data.xlsx`


In [ ]:
# Save to CSV and Excel
csv_path = "example_data.csv"
xlsx_path = "example_data.xlsx"

df_created.to_csv(csv_path, index=False)

# Excel writing uses an engine; in many environments openpyxl is available.
df_created.to_excel(xlsx_path, index=False)

print("Wrote files:", csv_path, "and", xlsx_path)


### 1B. Import from CSV

Key idea:
- `pd.read_csv(...)` reads a file and returns a DataFrame

Your turn:
- After loading, run `df_csv.shape` and interpret what it means.


In [ ]:
df_csv = pd.read_csv("example_data.csv")

print("CSV loaded.")
df_csv.head()


In [ ]:
# Basic exploration
df_csv.shape, df_csv.columns.tolist()


Your turn:
- What is one column that looks discrete?
- What is one column that looks continuous?

Run the next cell and scan the summary statistics.


In [ ]:
df_csv.describe(include="all")

### 1C. Import from Excel

Excel is common in business workflows.
- `pd.read_excel(...)` reads an Excel file and returns a DataFrame

Your turn:
- Verify `df_excel.equals(df_csv)` is `True` (or explain why it might not be).


In [ ]:
df_excel = pd.read_excel("example_data.xlsx")

print("Excel loaded.")
df_excel.head()


In [ ]:
# Check whether the two DataFrames match exactly
df_excel.equals(df_csv)

### 1D. Import from a URL (optional)

Many public datasets can be read directly from a URL using `pd.read_csv(url)`.

Notes:
- If your environment has no internet access, this may fail. That is OK.
- If it fails, continue; we will use the local dataset for the rest of the notebook.

Your turn:
- Predict: will this dataset have more rows or fewer rows than our created dataset?


In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

try:
    df_url = pd.read_csv(url)
    print("URL dataset loaded:", df_url.shape)
    display(df_url.head())
except Exception as e:
    print("URL import failed in this environment.")
    print("Error:", repr(e))
    df_url = None


### 1E. Choose one dataset to use as our “population”

We will use:
- the URL dataset if it loaded successfully, otherwise
- the locally created dataset

In this notebook, the word “population” means:
- the full dataset we currently have available

(That does not necessarily mean it is the full population in the real world.)


In [ ]:
if df_url is not None:
    df = df_url.copy()
    print("Using URL dataset as population.")
else:
    df = df_csv.copy()
    print("Using local dataset as population.")

df.shape, df.head()


## 2. Population vs sample

Definitions (in this notebook):
- Population: the full dataset `df`
- Sample: a subset of rows drawn from `df`

### Your turn
- If you take a sample of 50 rows from a population of 500 rows, how many rows are left out?
- Is a sample always representative? Why or why not?


In [ ]:
# Draw one sample of rows (without replacement by default)
sample_50 = df.sample(n=50, replace=False, random_state=42)

print("Population size:", len(df))
print("Sample size:", len(sample_50))
sample_50.head()


### 2A. Sampling with replacement vs without replacement

- Without replacement: you cannot pick the same row twice in one sample
- With replacement: the same row can appear multiple times

Your turn:
- Which one resembles “dealing cards from a deck”?
- Which one resembles “spinning a roulette wheel repeatedly”?


In [ ]:
sample_replace_50 = df.sample(n=50, replace=True, random_state=42)

# Check for duplicates
num_duplicates = sample_replace_50.duplicated().sum()
print("Duplicates in with-replacement sample:", num_duplicates)


## 3. Empirical distributions from data

An empirical distribution describes what we see in the data:
- For a numeric variable, a histogram is a common visualization
- For a categorical variable, a bar chart of counts or proportions is common

We will choose one numeric variable to study.

### Your turn
- Pick a numeric column you want to analyze.
- If you are not sure, use `df.select_dtypes("number").columns`.


In [ ]:
numeric_cols = df.select_dtypes("number").columns.tolist()
numeric_cols

Set the variable name in the next cell.

Suggested choices:
- If using the URL dataset: `total_bill` or `tip`
- If using the local dataset: `total_bill`, `tip`, or `tip_pct`


In [ ]:
# Choose a numeric variable from numeric_cols
var = "total_bill"  # change this if you want

assert var in df.columns, f"{var} not found in df columns"
df[var].head()


### 3A. Population histogram vs sample histogram

Your turn (prediction):
- In a small sample, do you expect the histogram to look “noisier” or “smoother” than the population histogram?


In [ ]:
# Population histogram
plt.figure()
plt.hist(df[var].dropna(), bins=30)
plt.title(f"Population distribution of {var}")
plt.xlabel(var)
plt.ylabel("Count")
plt.show()

# Sample histogram
plt.figure()
plt.hist(sample_50[var].dropna(), bins=30)
plt.title(f"Sample (n=50) distribution of {var}")
plt.xlabel(var)
plt.ylabel("Count")
plt.show()


## 4. Parameter vs statistic

- Parameter: a numerical feature of the population (in this notebook, the full dataset)
- Statistic: a numerical feature computed from a sample

Example:
- Population mean of `{var}` is a parameter
- Sample mean of `{var}` is a statistic

### Your turn
- Compute the population mean and the sample mean.
- Are they exactly equal? If not, why not?


In [ ]:
pop_mean = df[var].mean()
sample_mean = sample_50[var].mean()

pop_mean, sample_mean

## 5. Random variables (core concept)

A random variable (RV) is a numerical outcome of a random process.

Examples:
- Discrete RV: number of heads in 10 coin flips (values are countable)
- Continuous RV: time until a bus arrives (values vary continuously)

Key idea for this class:
- Many statistics are random variables (because they are computed from random samples)

### Your turn (short answers)
- Is the value of a single die roll a random variable?
- Is the sample mean of `{var}` a random variable?
Explain briefly.


### 5A. Discrete random variables

Discrete RVs take values from a countable set (often integers).

We will simulate:
- a fair die roll
- then estimate its empirical probabilities (an empirical PMF)

Your turn (prediction):
- If you roll a fair die 60 times, how many 6s do you expect (approximately)?


In [ ]:
# Simulate die rolls
n_rolls = 60
rolls = rng.integers(1, 7, size=n_rolls)  # values 1..6

rolls[:10], rolls.min(), rolls.max()


In [ ]:
# Empirical PMF (probability mass function) via proportions
values, counts = np.unique(rolls, return_counts=True)
pmf_emp = pd.Series(counts / counts.sum(), index=values).sort_index()

pmf_emp


In [ ]:
plt.figure()
plt.bar(pmf_emp.index.astype(str), pmf_emp.values)
plt.title("Empirical PMF of a die roll (n=60)")
plt.xlabel("Outcome")
plt.ylabel("Empirical probability")
plt.show()


### 5B. Continuous random variables

Continuous RVs can take infinitely many values in an interval.

We will simulate a Normal random variable using SciPy and look at:
- an empirical histogram (from simulated values)
- the theoretical PDF (probability density function)

Important conceptual point:
- For a continuous RV, P(X = exact value) = 0
- Probabilities come from intervals, like P(a ≤ X ≤ b)


### Understanding Probability in Continuous Random Variables

In a data science context, the distinction between discrete and continuous variables is fundamental to how we model uncertainty.

### 1. The Geometry of Probability
For a **discrete** random variable, we use a **Probability Mass Function (PMF)**, where we sum the "weights" of specific points. However, for a **continuous** random variable, probability is defined as the **area under the curve** of a **Probability Density Function (PDF)**.

* **Area = Width $\times$ Height:** To calculate a non-zero area, you must have a width (an interval).
* **The Single Point Problem:** An "exact value" has a width of zero.
* **Calculus Definition:** The probability that $X$ takes on a value in the interval $[a, b]$ is the integral of the PDF from $a$ to $b$. If $a = b$, the interval has no width:

$$P(X = a) = \int_{a}^{a} f(x) \, dx = 0$$



### 2. The Logic of Infinity
Consider a random variable representing the exact time a server takes to process a request. Between 120ms and 121ms, there are an **infinite** number of possible values (e.g., $120.5, 120.55, 120.552, \dots$).
* Because there are infinite possibilities, the probability of hitting one *exact* point with infinite decimal precision is $\frac{1}{\infty}$, which limits to zero.
* While $P(X = x) = 0$, this does not mean the outcome is "impossible." In continuous mathematics, we say the event has **measure zero**.

### 3. Practical Application in Data Science
In your work with data, we rarely measure "exact" values because of **sensor precision** or **rounding**.
* When we say a value is $5$, we usually mean it falls within the interval $[4.5, 5.5]$.
* We calculate the probability of falling within that **range**, which yields a non-zero result.

In [ ]:
# Simulate a continuous RV: Normal(0,1)
x = stats.norm.rvs(size=5000, random_state=42)

plt.figure()
plt.hist(x, bins=40, density=True)
plt.title("Empirical density from simulated Normal(0,1)")
plt.xlabel("x")
plt.ylabel("Density")
plt.show()


In [ ]:
# Overlay the theoretical PDF
grid = np.linspace(-4, 4, 400)
pdf = stats.norm.pdf(grid)

plt.figure()
plt.hist(x, bins=40, density=True)
plt.plot(grid, pdf)
plt.title("Histogram with theoretical Normal(0,1) PDF overlay")
plt.xlabel("x")
plt.ylabel("Density")
plt.show()


### 5C. Discrete vs continuous: PMF vs PDF

- PMF gives probabilities for exact values (discrete)
- PDF gives density; probability is area under the curve over an interval (continuous)

Your turn:
- For the Normal(0,1), compute P(-1 ≤ X ≤ 1) using the CDF.


In [ ]:
p_between = stats.norm.cdf(1) - stats.norm.cdf(-1)
p_between

### 5D. Random variables from real data

A column like `{var}` can be treated as a random variable:
- If you pick a random row from the population, you get a random value of `{var}`

Your turn:
- Decide whether `{var}` is best treated as discrete or continuous (and why).


## 6. Sampling distribution of a statistic (simulation)

We will build an empirical sampling distribution for the sample mean of `{var}`:

Process:
- Draw a sample of size n
- Compute the sample mean
- Repeat many times
- Plot the distribution of the sample means

Key idea:
- The sample mean is a statistic
- Because it comes from a random sample, it is a random variable

### Your turn (prediction)
- If we increase the sample size n, do you expect the sampling distribution to get wider or narrower?


When we increase the sample size $n$ from our `{var}` population, the sampling distribution of the mean becomes **narrower** and taller. This is a fundamental principle of statistical inference.

The Mathematical Intuition: Standard Error

The "width" of a sampling distribution is measured by the **Standard Error ($SE$)**. It represents the standard deviation of the sample means. The formula is:

$$SE = \frac{\sigma}{\sqrt{n}}$$

Where:
* $\sigma$ is the population standard deviation of `{var}`.
* $n$ is the sample size.

**The Relationship:**
Because $n$ is in the denominator, as $n$ increases, the $SE$ decreases. Mathematically, if you want to cut your error (the width) in half, you must increase your sample size by a factor of four ($2^2$).



In [ ]:
def sample_mean_once(df, var, n, rng):
    # Draw a sample and return the mean
    idx = rng.choice(len(df), size=n, replace=True)  # with replacement
    return df.iloc[idx][var].mean()

def sampling_distribution_means(df, var, n, reps, rng):
    means = np.empty(reps)
    for i in range(reps):
        means[i] = sample_mean_once(df, var, n, rng)
    return means

reps = 2000
n1 = 30
means_n1 = sampling_distribution_means(df, var, n=n1, reps=reps, rng=rng)

plt.figure()
plt.hist(means_n1, bins=40)
plt.title(f"Sampling distribution of mean({var}), n={n1}, reps={reps}")
plt.xlabel("Sample mean")
plt.ylabel("Count")
plt.show()

means_n1.mean(), means_n1.std()


### 6A. Compare sample sizes

Now repeat with a larger sample size and compare spreads.

Your turn:
- Change `n2` and rerun.
- Compare the standard deviations of the two sampling distributions.


In [ ]:
n2 = 200
means_n2 = sampling_distribution_means(df, var, n=n2, reps=reps, rng=rng)

plt.figure()
plt.hist(means_n2, bins=40)
plt.title(f"Sampling distribution of mean({var}), n={n2}, reps={reps}")
plt.xlabel("Sample mean")
plt.ylabel("Count")
plt.show()

print("Std dev with n =", n1, ":", means_n1.std())
print("Std dev with n =", n2, ":", means_n2.std())


## 7. Theoretical distributions in SciPy (PDF, CDF, PPF)

So far, we built distributions empirically from:
- observed data (population and samples), and
- simulation (sampling distributions)

Now we use mathematical probability models in SciPy.

We will use the standard Normal distribution as a reference model.

Key functions:
- `pdf(x)` density at x
- `cdf(x)` probability that X ≤ x
- `ppf(p)` value x such that P(X ≤ x) = p (percentile)

### Your turn
- Compute `cdf(1.5)` and interpret it in plain English.
- Compute the 95th percentile using `ppf(0.95)` and interpret it.


In [ ]:
cdf_15 = stats.norm.cdf(1.5)
p95 = stats.norm.ppf(0.95)

cdf_15, p95

## 8. Discrete distributions in SciPy: Binomial PMF

Example: number of heads in n coin flips.

Let X = number of heads in n flips.
- X is discrete
- X ~ Binomial(n, p)

We will compare:
- simulated counts of heads
- theoretical PMF from SciPy

### Your turn (prediction)
If n=20 and p=0.5, what value of X is most likely?


In [ ]:
n_flips = 20
p = 0.5
reps = 5000

# Simulate: each rep counts heads in n_flips
heads_counts = rng.binomial(n=n_flips, p=p, size=reps)

# Empirical PMF from simulation
vals, counts = np.unique(heads_counts, return_counts=True)
pmf_sim = pd.Series(counts / counts.sum(), index=vals)

# Theoretical PMF
k = np.arange(0, n_flips + 1)
pmf_theory = stats.binom.pmf(k, n=n_flips, p=p)

plt.figure()
plt.bar(pmf_sim.index, pmf_sim.values, alpha=0.7, label="Simulated")
plt.plot(k, pmf_theory, marker="o", linestyle="none", label="Theoretical PMF")
plt.title("Binomial(n=20, p=0.5): simulated vs theoretical")
plt.xlabel("Number of heads")
plt.ylabel("Probability")
plt.legend()
plt.show()


## 9. Connecting everything

Write short answers:

- What is a random variable?
- Give an example of a discrete random variable and its PMF.
- Give an example of a continuous random variable and explain why we use a PDF instead of a PMF.
- Why is the sample mean a random variable?
- What changed when we increased sample size in the sampling distribution simulation?

Optional extension:
- Try a different statistic (median) instead of mean and compare the sampling distribution.


## 10. Exit ticket (submit these)

Answer in 3–6 sentences each:

1. Explain the difference between an empirical distribution and a theoretical distribution.
2. Explain the difference between a parameter and a statistic.
3. Explain discrete vs continuous random variables with one example of each.
4. For a continuous random variable, why is P(X = exact value) = 0?
5. In this notebook, where did you see CDF and PPF used, and what questions do they answer?
